# fitcfg — GPU 2장 실측 (Kaggle 무료 T4 ×2)

학교 서버(MIG)로는 못 하는 **진짜 GPU 2장**에서 DDP · ZeRO-2 · ZeRO-3 · ZeRO-2+CPU offload 추천 설정을 실제로 학습해 봅니다.
확인하는 것: ① GPU당 메모리 예측 ② 추천 순위 = 실제 속도 순위 ③ GPU 1장 대비 확장 효율(지금은 문헌값 0.9 가정)

**준비 (오른쪽 패널 Settings)**
1. Accelerator → **GPU T4 x2**
2. Internet → **On** (휴대폰 인증된 계정 필요)
3. 위에서부터 셀 실행 → 40~60분. 끝나면 Output의 `results_kaggle.tar.gz`를 받아 capstone 폴더에 넣어 주세요.

In [ ]:
!nvidia-smi -L
!free -g | head -2

In [ ]:
%cd /kaggle/working
!rm -rf fitcfg && git clone -q https://github.com/dingdongri/fitcfg
%cd fitcfg
!pip install -q -e ".[hf]" bitsandbytes accelerate datasets "deepspeed>=0.14" 2>&1 | tail -n 3
# Kaggle 이미지의 torchao 0.10은 최신 transformers·peft LoRA와 충돌(ImportError) → 제거 (사용 안 함)
!pip uninstall -y -q torchao
!python -c "import torch,transformers,peft,deepspeed;print(torch.__version__,transformers.__version__,peft.__version__,deepspeed.__version__,torch.cuda.device_count())"

In [ ]:
!bash scripts/e2e_kaggle.sh 2>&1 | grep -vE 'Warning|warn' | tail -n 120

In [ ]:
!tar czf /kaggle/working/results_kaggle.tar.gz data/bundle_runs_kaggle.jsonl reports/e2e_kaggle_report.md 2>/dev/null; ls -la /kaggle/working/results_kaggle.tar.gz
!cat reports/e2e_kaggle_report.md 2>/dev/null | head -n 60
# 결과 원본을 로그에도 남긴다 (Output 파일을 못 받을 때 대비)
!echo "=== bundle_runs_kaggle.jsonl ==="; cat data/bundle_runs_kaggle.jsonl